# Factor 示例

读取同目录的 `factor.json` 配置，调用 `analyze_factors`
在 DolphinDB 中完成因子预处理、IC 和分组收益分析，并按需下载结果。

## Goal

1. 从 `factor.json` 加载数据集查询与分析参数。
2. 使用内置预处理（MAD 去极值、标准化、市值与行业中性化）。
3. 下载预处理因子表、IC / Rank IC 时间序列和市值加权分组收益。
4. 使用 `with` 管理结果持有的 DolphinDB session。

## Setup

在项目根目录运行 `uv run jupyter lab`。DolphinDB 连接参数从项目的
`.env` 或 `DOLPHIN_HOST`、`DOLPHIN_PORT`、`DOLPHIN_RUNTIME_USERNAME`、
`DOLPHIN_RUNTIME_PASSWORD` 环境变量读取。

DolphinDB 中需要已经存在 CoreData 统一因子表及示例日期范围的数据，
内置预处理还需要可用的股票行业元数据。

In [1]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
load_dotenv(project_root / ".env")
load_dotenv(project_root.parent / ".env")

from runtime import analyze_factors
from runtime.utils import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

2

## Steps

### 1. 加载配置

`factor.json` 与 `manage.py factor` 命令共用同一套字段；notebook 只取
分析参数，忽略其中的 `output_dir`。收益率和市值列会自动补入
`dataset_query.factors`。

In [2]:
config_path = project_root / "examples" / "factor.json"
config = json.loads(config_path.read_text(encoding="utf-8"))

run_arguments = {
    key: value
    for key, value in config.items()
    if key != "output_dir"
}

print("factor_columns:", run_arguments["factor_columns"])
print("return_columns:", run_arguments["return_columns"])
print("n_groups:", run_arguments["n_groups"])
print("preprocess:", run_arguments["preprocess"])

factor_columns: ['close', 'close_hfq']
return_columns: ['pct_chg', 'pct_chg_hfq']
n_groups: 5
preprocess: True


### 2. 执行分析并按需下载

`analyze_factors` 返回 `FactorAnalysisResult`。访问 `processed_data`、
`information_coefficients` 或 `all_group_returns` 时才从 session 下载；
退出 `with` 后 session 自动关闭。

In [3]:
with analyze_factors(**run_arguments) as factor_result:
    processed_data = factor_result.processed_data
    ic_tables = factor_result.information_coefficients
    group_return_tables = factor_result.all_group_returns

    print("session type:", type(factor_result.session).__name__)
    display(processed_data.head(10))
    for factor, ic in ic_tables.items():
        print(f"IC / Rank IC: {factor}")
        display(ic.head(10))
    for factor, group_returns in group_return_tables.items():
        print(f"分组收益: {factor}")
        display(group_returns.head(10))

print("session closed:", factor_result.closed)

2026-08-07 23:50:57.096 | INFO     | runtime.database.session:create_session:42 - DolphinDB: 127.0.0.1:8848
2026-08-07 23:50:57.187 | INFO     | runtime.apps.query.api:build_query_table:177 - session.run: 加载 query 模块
2026-08-07 23:50:57.287 | INFO     | runtime.database.session:has_session_variable:35 - session.run: 检查变量 coreFactorSourceData 是否存在
2026-08-07 23:50:57.289 | INFO     | runtime.apps.query.api:build_query_table:181 - session.run: 查询基础因子表 coreFactorSourceData
2026-08-07 23:50:57.794 | INFO     | runtime.apps.query.api:build_query_table:231 - session.run: 整理基础因子表 coreFactorSourceData
2026-08-07 23:50:57.796 | INFO     | runtime.apps.query.api:build_query_table:239 - session.run: 计算 coreFactorCOMPUTEDData 并生成 coreFactorFilteredData
2026-08-07 23:50:57.797 | INFO     | runtime.apps.query.api:build_query_table:252 - session.run: 投影 coreFactorFilteredData 生成 coreFactorInputData
2026-08-07 23:50:58.099 | SUCCESS  | runtime.utils.ts_api:get_stock_metadata:192 - Tushare Pro 初始化完成，共加

AttributeError: 'FactorAnalysisResult' object has no attribute 'information_coefficients'

## Checks

In [ ]:
factor = run_arguments["factor_columns"][0]

assert factor_result.closed
assert not processed_data.empty
assert set(ic_tables) == set(run_arguments["factor_columns"])
assert set(group_return_tables) == set(run_arguments["factor_columns"])
assert f"{factor}_group" in processed_data.columns
assert {"time", "code", factor}.issubset(processed_data.columns)

print("Factor example checks passed.")

## Next Steps

- 调整 `factor.json` 中的 `codes`、日期范围和 `n_groups`。
- 在 `factor_columns` 中加入多个因子，或使用 `dataset_query.derivatives`
  构造衍生因子。
- 设置 `preprocess=false` 时，需要在 `dataset_query` 中自行输出
  `{factor}_group` 分组列。
- 使用 `factor_result.information_coefficient(factor)` 或
  `factor_result.group_returns(factor)` 下载单个因子的结果。